# NeuroVision-X — Kaggle training driver

Thin driver. No training logic lives here: this notebook attaches the code and the data,
composes the real Hydra config, and calls `scripts/train.py::run_training`. Everything else
is in `src/neurovision/`, tested on the Mac's CPU.

**Before running:** notebook settings → GPU accelerator ON, internet ON (both need a
phone-verified account). Attach the preprocessed dataset. For a long run use
*Save Version → Save & Run All (Commit)*, never the interactive session.

Every cell below cell 1 fails immediately and with a readable message if a path is wrong —
the point is to find out in the first minute, not 40 minutes in.

Full workflow, including how to upload the dataset and chain sessions: `docs/kaggle_workflow.md`.

## 1. Session config — the only cell you edit

In [ ]:
# Set BEFORE torch is imported anywhere (this is the first cell that runs).
import os

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

REPO_URL   = "https://github.com/AmishhYadav/NeuroVision-X.git"
# UNCHANGED from session 1, deliberately. `git diff 7caacfa..main -- src/
# configs/ scripts/ requirements.txt` is EMPTY: every commit since then touched
# only the demo app, docs and this notebook. Resuming a run on a different
# training revision than the one that produced the checkpoint would make the
# run's provenance a range of commits rather than a commit, which is exactly
# what pinning exists to prevent. Do not "update to main" between sessions.
GIT_REF    = "7caacfa23078fac6358595777b9de3d9e3197914"
DATA_SLUG  = "amishyadav123/neurovision-brats-prep"

# --- RUN 2 of 3: neurovision -- RUN 2, SESSION 2 of 3 -------------------
# Session 1 (kernel `neurovision-run2-s1`, 10.34 GPU-h) was HEALTHY: epochs
# 0-35, clean max_hours stop, train/loss_epoch 0.5372 finite, grad-norm median
# 0.714 stable, best.pt at epoch 29 with val/dice_mean 0.8838, peak VRAM 6.16
# of 14.56 GiB. The fp16 entropy NaN that destroyed attempt 1 is gone: its
# `last.pt` was downloaded and checked here before this session was launched --
# zero non-finite parameter tensors and zero non-finite Adam moments.
#
# Session boundaries, from session 1's MEASURED cost (966 s/epoch, plus ~840 s
# on a validation epoch, val_interval 10) against max_hours 10.5:
#   session 1: epochs  0-35   CKPT_SLUG = None                  DONE, 10.34 h
#   session 2: epochs 36-~70  CKPT_SLUG = ...-run2-s1   ~10.3 h  <-- THIS RUN
#   session 3: epochs ~71-79  CKPT_SLUG = ...-run2-s2    ~2.7 h
CKPT_SLUG  = "amishyadav123/neurovision-run2-s1"

# Asserted in the resume cell, not merely printed. Attaching the WRONG
# checkpoint source is the one failure this notebook cannot detect by shape:
# attempt 1's abandoned NaN run also produced a `last.pt`, and resuming from it
# would train happily and produce a ruined 80-epoch run. Naming the epoch and
# the W&B id that the correct checkpoint MUST have turns that into a crash in
# the first minute. Set both to None only for a genuinely fresh run.
EXPECT_CKPT_EPOCH   = 35
EXPECT_WANDB_RUN_ID = "cc2l5j1c"

EXPERIMENT = None  # the experiment file already sets experiment_name

# Offline: a CLI-created kernel has no WANDB_API_KEY attached, and Kaggle
# Secrets cannot be set from kernel-metadata.json. The run id lives in the
# checkpoint, so all three sessions resume into ONE W&B run -- `cc2l5j1c`.
WANDB_MODE = "offline"

OVERRIDES  = [
    "+experiment=neurovision",
    # No gradient checkpointing: measured peak 6.16 GiB of 14.56 without it.
    # Whatever is set here must be set IDENTICALLY for
    # ablation_content_only_gate, or the P2 comparison differs by more than
    # the ambiguity signal it is meant to isolate.
    "data.num_workers=2",
]


## 2. Code + dependencies

Clone rather than `pip install git+...` alone: `configs/` and `scripts/` are not package data,
and Hydra needs the config tree on disk. The clone is then installed editable with `--no-deps`,
so `import neurovision` works without `PYTHONPATH` — same as local dev.

`torch`/`torchvision` are stripped from `requirements.txt` on purpose: the Kaggle image ships a
CUDA-matched build, and installing the pinned wheel over it silently loses the GPU. The assert
catches that, and a GPU that was never enabled, in ~30 seconds.

If `pip install -e` ever fails on `requires-python` (Kaggle moving off 3.11), replace that line
with `import sys; sys.path.insert(0, "/kaggle/working/repo/src")`.

In [ ]:
# Clone, THEN check out the pinned ref -- two steps, on purpose.
#
# `git clone -b` takes a BRANCH OR TAG NAME ONLY. Given a commit SHA it fails
# with `fatal: Remote branch <sha> not found in upstream origin`. This cost a
# session: cell 1 started pinning GIT_REF to a SHA (so a run's source revision
# stays recoverable), the clone silently failed, and the notebook died four
# lines later on a misleading `FileNotFoundError: .../repo/requirements.txt`.
#
# No `--depth 1` either: a shallow clone fetches only the branch tip, so any
# SHA that is not currently the tip cannot be checked out from it -- which is
# precisely the case for a run pinned to an earlier commit.
#
# subprocess(check=True), NOT `!git clone`: a failing `!` command does not stop
# a notebook cell, it prints to stderr and execution continues. That is what
# turned a one-line clone failure into an error four lines away. Every step
# below now raises where it actually fails.
import pathlib
import re
import subprocess
import sys

REPO_DIR = pathlib.Path("/kaggle/working/repo")
subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "-q", GIT_REF], check=True)
_head = subprocess.run(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
    capture_output=True, text=True, check=True,
).stdout.strip()
# Printed so the run's own log records the revision it executed -- the
# provenance that pinning GIT_REF exists to provide in the first place.
print(f"cloned {REPO_URL} at {_head}")
if not (REPO_DIR / "requirements.txt").is_file():
    raise FileNotFoundError(
        f"{REPO_DIR}/requirements.txt missing after clone + checkout of {GIT_REF}"
    )

# Build the Kaggle install list from requirements.txt minus its own
# `# kaggle-exclude:` line -- single source of truth, no second pinned file to
# drift. Everything excluded is either already in the Kaggle image (and ABI-
# linked to the rest of it) or dev-only.
_req = (REPO_DIR / "requirements.txt").read_text()
_excl = {w for m in re.findall(r"^#\s*kaggle-exclude:\s*(.+)$", _req, re.M) for w in m.split()}
_keep = [
    ln for ln in _req.splitlines()
    if ln.strip() and not ln.lstrip().startswith("#")
    and re.split(r"[=<>~!\[]", ln.strip())[0].strip() not in _excl
]
print("installing:", " ".join(_keep))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_keep], check=True)

In [ ]:
import torch

# sys.path, NOT `pip install -e`. Kaggle runs Python 3.12 and pyproject.toml
# pins requires-python = ">=3.11,<3.12", so pip REFUSES the editable install
# ("Package 'neurovision-x' requires a different Python"). A `!pip` failure
# does not stop a notebook cell, so that error scrolled past and a previous run
# died four cells later on ModuleNotFoundError. sys.path needs no metadata
# check and works on any interpreter.
sys.path.insert(0, "/kaggle/working/repo/src")
sys.path.insert(0, "/kaggle/working/repo/scripts")
import neurovision  # noqa: F401  -- verify HERE, not four cells later
import scipy.ndimage  # noqa: F401 -- canary: breaks if our numpy pin overwrote Kaggle's

assert torch.cuda.is_available(), "No CUDA: GPU accelerator off, or pip replaced Kaggle's CUDA torch build."
_name = torch.cuda.get_device_name(0)
_cap = "sm_%d%d" % torch.cuda.get_device_capability(0)
# is_available() is NOT sufficient, and this is not hypothetical: on a Kaggle
# P100 it returns True while every kernel launch fails, because the stock torch
# build no longer targets sm_60 (min sm_70). Only executing something is honest.
try:
    (torch.randn(64, 64, device="cuda") @ torch.randn(64, 64, device="cuda")).sum().item()
except Exception as exc:
    raise RuntimeError(
        f"{_name} ({_cap}) reports CUDA available but cannot run a kernel: {exc}\n"
        "Set machine_shape=NvidiaTeslaT4 (sm_75); the P100 is sm_60."
    ) from exc

import monai
import numpy as np

print(f"{_name}  {_cap}")
print(f"torch {torch.__version__} | numpy {np.__version__} | monai {monai.__version__} "
      f"| python {sys.version.split()[0]}")

## 3. Environment — W&B

`WANDB_MODE` in cell 1 picks one of three:

- **`"online"`** — needs `WANDB_API_KEY` as a Kaggle Secret, added under Add-ons → Secrets **and
  attached to this notebook**. A secret on your account but not attached to this kernel fails
  with `No user secrets exist for kernel id <id> and label <label>`. The label is exact and
  case-sensitive; set `SECRET_LABEL` to whatever you actually named it.
- **`"offline"`** — no secret needed. The full run is written to `/kaggle/working/wandb/` and
  saved with the notebook output; `wandb sync <dir>` uploads it later with every metric intact.
- **`"disabled"`** — no logging at all.

Never paste the key into a cell: committed notebook versions are stored with their source.

In [ ]:
import os

# Exact, case-sensitive label of the Kaggle Secret. Must match what you named it
# in Add-ons -> Secrets AND be attached to THIS notebook.
SECRET_LABEL = "WANDB_API_KEY"

# Only "online" needs a Kaggle Secret. "offline" writes the complete run to
# /kaggle/working/wandb/ with no API key at all -- it lands in the notebook
# output, and `wandb sync <dir>` from your Mac uploads it afterwards with the
# curves intact. Use it when the secret is unavailable; it is NOT a downgrade
# in what gets recorded, only in when it appears in the dashboard.
if WANDB_MODE == "online":
    from kaggle_secrets import UserSecretsClient

    # Not wrapped in try/except: a missing secret on a real run means losing the
    # run's curves, so it must stop here, not 11 hours in.
    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret(SECRET_LABEL)
    print(f"W&B online (secret label {SECRET_LABEL!r})")
elif WANDB_MODE == "offline":
    os.environ["WANDB_DIR"] = "/kaggle/working"
    OVERRIDES = [*OVERRIDES, "wandb.mode=offline"]
    print("W&B OFFLINE -- run written to /kaggle/working/wandb/. "
          "Upload later with: wandb sync <that dir>")
else:
    OVERRIDES = [*OVERRIDES, "wandb.mode=disabled"]
    print("W&B DISABLED -- nothing will be logged.")

## 4. Resolve the attached dataset

Layout is what `scripts/package_for_kaggle.py` builds: `preprocessed/<case>/{image,label}.npy`,
`metadata.csv`, `splits.yaml`. A wrong or unattached dataset raises here, listing what *is*
mounted so the fix is obvious.

In [ ]:
import shutil
from pathlib import Path

# Discovered, not assumed. A dataset does NOT reliably mount at
# /kaggle/input/<slug>: an earlier run of this notebook found the whole of
# /kaggle/input to be just ['datasets'], i.e. one level deeper than the
# documented layout. So look for the shape we need -- a directory holding both
# preprocessed/ and splits.yaml -- across the first few levels, rather than
# hardcoding a path Kaggle is free to change.
_roots = [Path("/kaggle/input")]
_hits = sorted(
    {
        p.parent
        for pat in ("splits.yaml", "*/splits.yaml", "*/*/splits.yaml", "*/*/*/splits.yaml")
        for r in _roots
        for p in r.glob(pat)
        if (p.parent / "preprocessed").is_dir()
    }
)
if len(_hits) != 1:
    _tree = sorted(str(p.relative_to("/kaggle/input")) for p in Path("/kaggle/input").glob("*/*"))
    raise FileNotFoundError(
        f"Expected exactly one dataset with preprocessed/ + splits.yaml under /kaggle/input, "
        f"found {[str(h) for h in _hits]}. Attach {DATA_SLUG}. Present: {_tree[:20]}"
    )
DATA = _hits[0]
PREP, SPLITS = DATA / "preprocessed", DATA / "splits.yaml"
n_cases = sum(1 for p in PREP.iterdir() if p.is_dir())
if n_cases == 0:
    raise FileNotFoundError(f"{PREP} exists but holds no case directories.")
print(n_cases, "preprocessed cases at", PREP)

## 5. Resume

`/kaggle/input` is read-only and `save_checkpoint` must write, so the previous session's
`last.pt` is copied into `/kaggle/working/checkpoints` first. `select_resume_checkpoint` then
finds it there on its own — the training call is identical for a fresh run and a resume.

With `CKPT_SLUG` set, anything other than exactly one `last.pt` raises. Missing it would not
error during training, it would just silently restart from epoch 0 — the expensive failure this
cell exists to prevent.

In [ ]:
CKPT_DIR = Path("/kaggle/working/checkpoints")
CKPT_DIR.mkdir(parents=True, exist_ok=True)
if CKPT_SLUG:
    def _find_ckpt(fname):
        """Every distinct <fname> under /kaggle/input, as ORIGINAL paths.

        EXPLICIT DEPTHS, not just `**`. pathlib's `**` does not recurse into
        symlinked directories, and how Kaggle materialises an attached kernel
        output is not something this notebook gets to assume -- the data cell
        above already had to stop assuming /kaggle/input/<slug> after a real run
        found everything one level deeper. A `**` that quietly traverses nothing
        would raise "found []" and burn the session launch.

        Deduped BY realpath (one source mounted at two paths is one checkpoint,
        not an ambiguity) but RETURNING the original path, because the caller
        looks for a sibling file next to it. Returning the resolved path would
        point .parent at the symlink target's directory instead of the mount --
        which is how an earlier version of this cell looked for best.pt in
        entirely the wrong place.
        """
        pats = [fname, f"*/{fname}", f"*/*/{fname}", f"*/*/*/{fname}",
                f"*/*/*/*/{fname}", f"*/*/*/*/*/{fname}", f"**/{fname}"]
        by_real = {}
        for pat in pats:
            for p in Path("/kaggle/input").glob(pat):
                by_real.setdefault(os.path.realpath(p), p)
        items = sorted(by_real.values(), key=str)
        # Prefer hits whose path names the checkpoint source we actually asked
        # for. With several sources attached this distinguishes "the checkpoint"
        # from "some other last.pt" instead of failing on a resolvable
        # ambiguity. Falls back to the full list if the name appears nowhere, so
        # a Kaggle mount-layout change degrades to the old behaviour.
        named = [p for p in items if CKPT_SLUG.split("/")[-1] in str(p)]
        return named or items

    hits = _find_ckpt("last.pt")
    if len(hits) != 1:
        mounted = sorted(str(q.relative_to("/kaggle/input")) for q in Path("/kaggle/input").glob("*/*"))
        raise FileNotFoundError(
            f"CKPT_SLUG={CKPT_SLUG!r}: want exactly one last.pt under /kaggle/input, found "
            f"{[str(h) for h in hits]}. Attach the checkpoint source, or set CKPT_SLUG=None "
            f"to start fresh. Mounted: {mounted[:20]}"
        )
    # /kaggle/input is read-only and save_checkpoint must write, so the file is
    # copied into the writable dir. find_resume_checkpoint then picks it up on
    # its own -- the training call is identical for a fresh run and a resume.
    shutil.copy2(hits[0], CKPT_DIR / "last.pt")

    # best.pt travels too, when the previous session has one. Nothing in the
    # trainer reads it -- resume goes through last.pt -- but it is the
    # checkpoint the paper reports from, and save_checkpoint only rewrites it
    # when validation IMPROVES. On the final session of a run that peaked
    # earlier, no new best occurs, so without this copy the last session's
    # output holds no best.pt at all and the run's reportable checkpoint is
    # stranded in whichever earlier session happened to produce it. Measured on
    # run 2: it peaked at epoch 69 in session 2, session 3 trained 73-79 without
    # improving, and the verification cell then failed on the missing file --
    # marking a session that had trained all 80 epochs as ERROR.
    _bests = _find_ckpt("best.pt")
    _sib = [p for p in _bests if p.parent == hits[0].parent]  # the matching one
    _best_src = (_sib or _bests or [None])[0]
    if _best_src is not None:
        shutil.copy2(_best_src, CKPT_DIR / "best.pt")
        print(f"carried forward {_best_src}")
    else:
        print("no best.pt alongside the resume checkpoint (fine for a run that "
              "has not validated yet; check an earlier session otherwise)")

    import torch as _t
    _ck = _t.load(CKPT_DIR / "last.pt", weights_only=True, map_location="cpu")
    print(f"resuming from {hits[0]}")
    print(f"  epoch={_ck['epoch']} -> will start at {_ck['epoch']+1}, "
          f"best {_ck['best_metric_name']}={_ck['best_metric']:.4f}, wandb_run_id={_ck.get('wandb_run_id')}")

    # Identity checks. A checkpoint from the wrong run loads perfectly and
    # trains perfectly; the damage only shows up 10 hours later in a run that
    # is not the run you meant. These cost milliseconds.
    if EXPECT_CKPT_EPOCH is not None and _ck["epoch"] != EXPECT_CKPT_EPOCH:
        raise RuntimeError(
            f"checkpoint is at epoch {_ck['epoch']}, expected {EXPECT_CKPT_EPOCH}. "
            f"Wrong session's output attached, or the previous session did not end "
            f"where its log says."
        )
    if EXPECT_WANDB_RUN_ID is not None and _ck.get("wandb_run_id") != EXPECT_WANDB_RUN_ID:
        raise RuntimeError(
            f"checkpoint carries wandb_run_id={_ck.get('wandb_run_id')!r}, expected "
            f"{EXPECT_WANDB_RUN_ID!r}. This is a DIFFERENT run -- attempt 1 (the NaN run) "
            f"also produced a last.pt. Resuming it would waste the whole session."
        )

    # The failure that cost 10.5 GPU-h once already. NaN weights train without
    # raising anything, so the only honest check is to look. ~2 s over 34.9M
    # parameters, against a 10-hour session.
    _bad = [k for k, v in _ck["model_state_dict"].items()
            if v.is_floating_point() and not _t.isfinite(v).all()]
    _bad_opt = [i for i, s in _ck["optimizer_state_dict"]["state"].items()
                if any(_t.is_tensor(t) and t.is_floating_point() and not _t.isfinite(t).all()
                       for t in s.values())]
    if _bad or _bad_opt:
        raise RuntimeError(
            f"checkpoint contains non-finite values: {len(_bad)} parameter tensors "
            f"(e.g. {_bad[:3]}), {len(_bad_opt)} optimizer states. Do NOT resume this."
        )
    print(f"  finite: {len(_ck['model_state_dict'])} parameter tensors, "
          f"{len(_ck['optimizer_state_dict']['state'])} optimizer states, no NaN/Inf")
    del _ck


## 6. Compose config and train

`hydra.compose` with CLI-style overrides — the same strings `python scripts/train.py a=b` would
take. Calling `run_training` in-process (instead of shelling out) keeps the traceback in the
notebook and lets the cells above hand it already-validated paths.

The log's first line says `FRESH:` or `RESUME: ... from epoch N`. Check it. `max_hours: 10.5`
(set in `configs/experiment/_baseline_common.yaml`) stops the run cleanly before Kaggle's
12-hour kill, leaving ~50 minutes of margin for the worst case — a budget check that passes on
the mean epoch duration and is then followed by a validation epoch costing ~0.21 h more than
that mean.

In [ ]:
import hydra

# sys.path for both src/ and scripts/ was set in the install cell, so this
# import cannot be the first place a missing package shows up.
from train import run_training

overrides = [f"data.root_dir={DATA}", f"data.preprocessing.out_dir={PREP}", f"data.splits.path={SPLITS}",
             f"training.checkpoint.dir={CKPT_DIR}"]
# experiment_name is passed ONLY when EXPERIMENT is set. An explicit
# experiment_name= override always beats a config group's own value, whatever
# the ordering -- Hydra applies group additions during composition and value
# overrides afterwards. So passing it unconditionally would silently rename an
# `+experiment=overfit2` run to EXPERIMENT, sending its checkpoints to
# outputs/<EXPERIMENT> and labelling its W&B run as that experiment. Set
# EXPERIMENT = None whenever OVERRIDES contains a `+experiment=` entry.
if EXPERIMENT is not None:
    overrides.append(f"experiment_name={EXPERIMENT}")
overrides += OVERRIDES

with hydra.initialize_config_dir(version_base="1.3", config_dir="/kaggle/working/repo/configs"):
    cfg = hydra.compose("config", overrides=overrides)
print("experiment_name =", cfg.experiment_name, "| epochs =", cfg.training.epochs)
metrics = run_training(cfg)

## 7. Verify the session output

Training already writes into `/kaggle/working/checkpoints`, which is the only path that survives
into the committed version's output — so there is nothing to copy, only to verify. Duplicating a
754 MB SwinUNETR checkpoint elsewhere under `/kaggle/working` would just eat the ~20 GB quota.

Attach this notebook version's output as input to the next session and set `CKPT_SLUG` to it.

In [ ]:
# Peak VRAM, reported rather than estimated. Three separate pre-run estimates
# of this model's memory were wrong (16 GB assumed capacity vs 14.56 actual,
# a 0.55 AMP factor that is really ~0.75+, and a per-patch figure read as a
# per-step one), so the run itself is now the source of truth. max_memory_*
# are process-lifetime peaks and survive the training call in the same kernel.
import math

_total = torch.cuda.get_device_properties(0).total_memory / 2**30
print(
    f"peak VRAM: {torch.cuda.max_memory_allocated() / 2**30:.2f} GiB allocated, "
    f"{torch.cuda.max_memory_reserved() / 2**30:.2f} GiB reserved, "
    f"of {_total:.2f} GiB total"
)

# last.pt is REQUIRED -- without it the next session has nothing to resume from.
_last = CKPT_DIR / "last.pt"
if not _last.is_file():
    raise FileNotFoundError(f"{_last} missing — nothing to carry into the next session.")
_final_epoch = torch.load(_last, weights_only=True)["epoch"]
print("last.pt", f"{_last.stat().st_size / 2**20:.0f} MB  epoch={_final_epoch}")

# best.pt is EXPECTED BUT NOT REQUIRED, and raising on it was a real bug: it
# marked run 2 session 3 as ERROR after that session had successfully trained
# all 80 epochs. save_checkpoint only writes best.pt when validation IMPROVES,
# so a final session on a run that peaked earlier legitimately produces none.
# The resume cell now carries the previous best.pt forward, which covers the
# normal case; this stays non-fatal because the alternative is failing a
# session whose real work is complete and whose checkpoints are already on
# disk. Report it, do not raise on it.
_best = CKPT_DIR / "best.pt"
if _best.is_file():
    _be = torch.load(_best, weights_only=True)["epoch"]
    print("best.pt", f"{_best.stat().st_size / 2**20:.0f} MB  epoch={_be}")
else:
    _be = None
    print("best.pt ABSENT — no validation improved during this session and none was "
          "carried forward. The run's best checkpoint lives in an earlier session's "
          "output; find it there before evaluating.")
print(metrics)

# One grep-able health line, printed LAST. The next session is launched from a
# watcher that reads this log, and "the kernel reached COMPLETE" says nothing
# about whether the run diverged -- attempt 1 completed cleanly while training
# on NaN for 29 epochs. Printed rather than raised on purpose: a raise here
# would mark the commit failed AFTER the checkpoint was already safely written.
# (Kaggle does still persist a failed version's output -- measured on run 2
# session 3 -- but the run log then ends in a traceback instead of a verdict,
# which is strictly worse for an unattended chain.)
_nan = [k for k, v in metrics.items() if isinstance(v, float) and not math.isfinite(v)]
_status = "NAN" if _nan else "OK"
print(f"NVX_HEALTH: {_status} | epoch={_final_epoch} | best_epoch={_be} | "
      f"loss={metrics.get('train/loss_epoch')} | "
      f"grad_norm_median={metrics.get('train/grad_norm_median')} | "
      f"nonfinite={_nan}")
